In [1]:
# =============================================================================
# DATA AGENT V1
# =============================================================================
#
# ARQUITETURA
#
# Question
#     ↓
# Schema
#     ↓
# Text-to-SQL
#     ↓
# Execute SQL
#     ↓
# Response
#
# Objetivo:
#
# Construir a primeira versão funcional de um Data Agent
# para consultas em dados de produção de petróleo.
#
# Nesta versão NÃO teremos:
#
# - SQL Validator
# - Memory
# - Analytics Layer
# - LangGraph
# - Multi-Agent
# - LangSmith
#
# Foco:
#
# Entender claramente cada camada.
#
# =============================================================================


# =============================================================================
# STEP 1 - IMPORTS
# =============================================================================

# SQLite para banco local
import sqlite3

# Pandas para consultas e visualização
import pandas as pd

# Modelo local executado pelo Ollama
from langchain_ollama import ChatOllama


# =============================================================================
# STEP 2 - DATABASE CREATION
# =============================================================================
#
# Nesta etapa criamos uma base simples para testes.
#
# Em produção esta etapa normalmente NÃO existiria,
# pois o banco já estaria pronto.
#
# =============================================================================

# Cria conexão com banco SQLite
conn = sqlite3.connect("oil.db")

# Cursor para execução de comandos SQL
cursor = conn.cursor()


# -----------------------------------------------------------------------------
# Remove tabela antiga
# -----------------------------------------------------------------------------

cursor.execute("""
DROP TABLE IF EXISTS well_production
""")


# -----------------------------------------------------------------------------
# Cria tabela de produção
# -----------------------------------------------------------------------------

cursor.execute("""
CREATE TABLE well_production (

    well_name TEXT,
    field_name TEXT,
    production_date TEXT,

    oil_bbl REAL,
    gas_mscf REAL,
    water_bbl REAL,

    hours_on REAL
)
""")


# -----------------------------------------------------------------------------
# Dados de exemplo
# -----------------------------------------------------------------------------

rows = [

    ("WELL-A1","FIELD-X","2026-06-01",1200,800,300,24),
    ("WELL-A2","FIELD-X","2026-06-01",900,600,500,24),
    ("WELL-B1","FIELD-Y","2026-06-01",1500,1100,200,24),

    ("WELL-A1","FIELD-X","2026-06-02",1250,820,320,24),
    ("WELL-A2","FIELD-X","2026-06-02",920,620,510,24),
    ("WELL-B1","FIELD-Y","2026-06-02",1520,1120,210,24),

    ("WELL-A1","FIELD-X","2026-06-03",1230,810,310,24),
    ("WELL-A2","FIELD-X","2026-06-03",910,610,505,24),
    ("WELL-B1","FIELD-Y","2026-06-03",1550,1150,220,24)

]


# -----------------------------------------------------------------------------
# Insere registros
# -----------------------------------------------------------------------------

cursor.executemany("""
INSERT INTO well_production
VALUES (?,?,?,?,?,?,?)
""", rows)

conn.commit()

print("Banco criado com sucesso")


# =============================================================================
# STEP 3 - SCHEMA LAYER
# =============================================================================
#
# Esta camada é responsável por descobrir a estrutura
# do banco e fornecer contexto para o LLM.
#
# =============================================================================

schema_df = pd.read_sql(
    "PRAGMA table_info(well_production)",
    conn
)

# Extrai apenas nomes das colunas
schema_text = "\n".join(
    schema_df["name"].tolist()
)

print("\n")
print("=" * 80)
print("SCHEMA")
print("=" * 80)
print(schema_text)


# =============================================================================
# STEP 4 - LLM INITIALIZATION
# =============================================================================
#
# Modelo responsável por converter linguagem natural
# em SQL.
#
# =============================================================================

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)


# =============================================================================
# STEP 5 - USER QUESTION
# =============================================================================
#
# Em uma aplicação real esta pergunta viria:
#
# - Chat
# - API
# - Streamlit
# - FastAPI
# - Interface Web
#
# =============================================================================

question = """
Qual poço teve maior produção acumulada de óleo?
"""


print("\n")
print("=" * 80)
print("PERGUNTA")
print("=" * 80)
print(question)


# =============================================================================
# STEP 6 - TEXT TO SQL LAYER
# =============================================================================
#
# Esta é a camada principal do agente.
#
# Responsabilidade:
#
# Linguagem Natural
#      ↓
# SQL
#
# =============================================================================

prompt = f"""
Você é um especialista em SQL.

Sua tarefa é converter perguntas em SQL.

Tabela:

well_production

Colunas disponíveis:

{schema_text}

REGRAS:

1) Quando o usuário falar:

produção acumulada

utilize:

SUM(oil_bbl)

2) Retorne SOMENTE SQL.

3) Não explique nada.

4) Não utilize markdown.

5) Não utilize blocos ```sql

Exemplo:

SELECT
    well_name,
    SUM(oil_bbl) AS total_oil
FROM well_production
GROUP BY well_name
ORDER BY total_oil DESC
LIMIT 1

Pergunta:

{question}
"""

# Envia prompt para o modelo
response = llm.invoke(prompt)

# Extrai SQL gerado
sql = response.content.strip()


print("\n")
print("=" * 80)
print("SQL GERADO")
print("=" * 80)
print(sql)


# =============================================================================
# STEP 7 - EXECUTION LAYER
# =============================================================================
#
# Responsabilidade:
#
# SQL
#   ↓
# Resultado
#
# =============================================================================

print("\n")
print("=" * 80)
print("EXECUTANDO SQL")
print("=" * 80)

try:

    # Executa SQL gerado pelo modelo
    result_df = pd.read_sql(
        sql,
        conn
    )

    print(result_df)

except Exception as e:

    print("\nERRO AO EXECUTAR SQL")
    print(e)

    result_df = None


# =============================================================================
# STEP 8 - RESPONSE LAYER
# =============================================================================
#
# Responsabilidade:
#
# Resultado SQL
#       ↓
# Resposta para usuário
#
# Nesta V1 faremos apenas uma resposta simples.
#
# =============================================================================

print("\n")
print("=" * 80)
print("RESPONSE")
print("=" * 80)

if result_df is not None and len(result_df) > 0:

    print("Consulta executada com sucesso.")
    print()
    print(result_df.to_string(index=False))

else:

    print("Nenhum resultado encontrado.")


# =============================================================================
# STEP 9 - CLOSE CONNECTION
# =============================================================================

conn.close()

print("\nConexão encerrada.")

Banco criado com sucesso


SCHEMA
well_name
field_name
production_date
oil_bbl
gas_mscf
water_bbl
hours_on


PERGUNTA

Qual poço teve maior produção acumulada de óleo?



SQL GERADO
SELECT 
    well_name, 
    SUM(oil_bbl) AS total_oil
FROM 
    well_production
GROUP BY 
    well_name
ORDER BY 
    total_oil DESC
LIMIT 1


EXECUTANDO SQL
  well_name  total_oil
0   WELL-B1     4570.0


RESPONSE
Consulta executada com sucesso.

well_name  total_oil
  WELL-B1     4570.0

Conexão encerrada.
